# Experiment B — Is the cross-country drop caused by heterogeneity between national markets?

For every country with enough records, Random Forest (Experiment B, `criterion='entropy'`, RandomOverSampler inside training data) is evaluated in three settings:

| Setting | Trained on | Evaluated on |
|---|---|---|
| **(a) within** | that country only (stratified k=5) | that country |
| **(b) pooled, included** | all countries, stratified k=10 as in the paper | that country's out-of-fold records |
| **(c) pooled, excluded** | all *other* countries | that country |

How to read it:

- **(a) ≈ (b) clearly above (c)** → the country has learnable patterns, but they differ from the rest of the EU: **heterogeneity between national markets confirmed**.
- **(a) low as well** → the country is simply hard to predict with these features; the cross-country drop would not be explained by heterogeneity.
- **(b) above (a)** → pooling helps that country (more data outweighs the differences).

A last section adds all 30 `ISO_COUNTRY_CODE` dummies to the 16 predictors (the paper's feature selection kept only `lu` and `si`) to check whether letting the model know the country improves pooled performance.

Expected runtime: about 10–12 minutes. Results are saved to `country_shift_results.json` (copy in `results/review_2/`) and feed Table 19 of the revised manuscript.

In [ ]:
# ── COLAB SETUP ─────────────────────────────────────────────────────────
import os, shutil
DRIVE_PARQUET = '/content/drive/MyDrive/fci-datos/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'
LOCAL_PARQUET = '/content/part-00000-b5be9405-11c0-4c04-a771-bac38e228ad8-c000.snappy.parquet'

if not os.path.exists(LOCAL_PARQUET):
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        shutil.copy(DRIVE_PARQUET, LOCAL_PARQUET)
        print(f'Parquet copied from Drive to {LOCAL_PARQUET}')
    except Exception as e:
        print(f'Drive mount failed ({e}); upload the parquet manually:')
        from google.colab import files
        uploaded = files.upload()
        os.rename(list(uploaded.keys())[0], LOCAL_PARQUET)
        print(f'Parquet ready at {LOCAL_PARQUET}')
else:
    print(f'Parquet already available at {LOCAL_PARQUET}')

!pip install imbalanced-learn -q
print('Setup complete.')

In [ ]:
import pandas as pd
import numpy as np
import time, json, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.base import clone
from imblearn.over_sampling import RandomOverSampler

df_raw = pd.read_parquet(LOCAL_PARQUET)
for col in df_raw.select_dtypes(include='string').columns:
    df_raw[col] = df_raw[col].astype(object)
for col in ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS', 'NUMBER_TENDERS_SME', 'YEAR']:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype(int)

df_raw['GROUP_CPV'] = df_raw['CPV'].astype(str).str[:2]
cat_cols = ['B_MULTIPLE_CAE', 'B_ON_BEHALF', 'CAE_TYPE', 'MAIN_ACTIVITY',
            'ISO_COUNTRY_CODE', 'TYPE_OF_CONTRACT', 'GROUP_CPV']
for col in cat_cols:
    df_raw[col] = df_raw[col].astype(str)

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_arr = ohe.fit_transform(df_raw[cat_cols])
num_cols = ['NUMBER_AWARDS', 'LOTS_NUMBER', 'NUMBER_OFFERS']
df_proc = pd.concat([df_raw[num_cols].reset_index(drop=True),
                     pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(cat_cols))], axis=1)

df_raw['SME_WIN'] = df_raw['B_CONTRACTOR_SME'].astype(str).str.contains('y', case=False, na=False).astype(int)

used_features_B2 = [
    "B_MULTIPLE_CAE_n", "B_ON_BEHALF_n",
    "GROUP_CPV_45", "GROUP_CPV_33", "GROUP_CPV_15",
    "TYPE_OF_CONTRACT_w", "MAIN_ACTIVITY_health", "ISO_COUNTRY_CODE_si",
    "NUMBER_AWARDS", "LOTS_NUMBER", "NUMBER_OFFERS",
    "CAE_TYPE_3", "MAIN_ACTIVITY_general public\\services",
    "CAE_TYPE_4", "CAE_TYPE_5", "ISO_COUNTRY_CODE_lu"
]
X = df_proc[used_features_B2].astype(float).to_numpy()
y = df_raw['SME_WIN'].reset_index(drop=True).to_numpy()
paises = df_raw['ISO_COUNTRY_CODE'].reset_index(drop=True).to_numpy()
assert X.shape[1] == 16

rf_B = RandomForestClassifier(criterion='entropy', max_features='sqrt', bootstrap=True,
                              max_samples=2/3, n_estimators=100, n_jobs=-1, random_state=42)
ros = RandomOverSampler(random_state=42)
print('X', X.shape, '| overall SME_WIN rate', round(y.mean(), 3))

## 1. SME_WIN rate per country (descriptive evidence of heterogeneity)

In [ ]:
desc = (pd.DataFrame({'country': paises, 'SME_WIN': y})
          .groupby('country')['SME_WIN'].agg(n='size', sme_win_rate='mean', n_neg=lambda s: int((s == 0).sum()))
          .sort_values('n', ascending=False))
desc['sme_win_rate'] = desc['sme_win_rate'].round(3)
print(desc.to_string())

MIN_N, MIN_NEG = 5000, 300
COUNTRIES = [c for c in desc.index if desc.loc[c, 'n'] >= MIN_N and desc.loc[c, 'n_neg'] >= MIN_NEG]
print(f'\nCountries analysed (n >= {MIN_N}, SME_WIN=0 >= {MIN_NEG}):', COUNTRIES)

## 2. Helpers

In [ ]:
def scores(y_true, y_pred, y_prob):
    out = {'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0))}
    out['auc'] = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else float('nan')
    return out

def majority_f1(y_train, y_test):
    maj = np.bincount(y_train).argmax()
    return float(f1_score(y_test, np.full(len(y_test), maj), average='macro', zero_division=0))

def fit_predict(X_tr, y_tr, X_te):
    m = clone(rf_B)
    Xb, yb = ros.fit_resample(X_tr, y_tr)
    m.fit(Xb, yb)
    return m.predict(X_te), m.predict_proba(X_te)[:, 1]

## 3. Setting (b): pooled stratified k=10 (as in the paper), out-of-fold predictions
Should reproduce the paper's AUC ≈ 0.7100 overall.

In [ ]:
t0 = time.time()
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_pred = np.zeros(len(y), dtype=int)
oof_prob = np.zeros(len(y))
for tr, te in skf.split(X, y):
    oof_pred[te], oof_prob[te] = fit_predict(X[tr], y[tr], X[te])
overall = scores(y, oof_pred, oof_prob)
print(f'Pooled OOF overall: macro-F1={overall["f1_macro"]:.4f}  AUC={overall["auc"]:.4f}  ({(time.time()-t0)/60:.1f} min)')

## 4. Settings (a) within-country and (c) pooled-excluded, per country

In [ ]:
rows = []
for c in COUNTRIES:
    t0 = time.time()
    mask = paises == c
    Xc, yc = X[mask], y[mask]

    # (a) within-country stratified k=5, pooled out-of-fold predictions
    skf5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pa, qa = np.zeros(len(yc), dtype=int), np.zeros(len(yc))
    for tr, te in skf5.split(Xc, yc):
        pa[te], qa[te] = fit_predict(Xc[tr], yc[tr], Xc[te])
    a = scores(yc, pa, qa)

    # (b) pooled, country included: paper's OOF predictions restricted to this country
    b = scores(yc, oof_pred[mask], oof_prob[mask])

    # (c) pooled, country excluded: train on all other countries
    pc, qc = fit_predict(X[~mask], y[~mask], Xc)
    cx = scores(yc, pc, qc)

    row = {'country': c, 'n': int(mask.sum()), 'sme_win_rate': float(yc.mean()),
           'majority_f1_macro': majority_f1(yc, yc),
           'a_within': a, 'b_pooled_included': b, 'c_pooled_excluded': cx}
    rows.append(row)
    print(f'{c.upper():<3} n={mask.sum():>6,} rate={yc.mean():.3f} | AUC  a={a["auc"]:.3f}  b={b["auc"]:.3f}  c={cx["auc"]:.3f} '
          f'| macro-F1  a={a["f1_macro"]:.3f}  b={b["f1_macro"]:.3f}  c={cx["f1_macro"]:.3f}  ({time.time()-t0:.0f}s)')

res = pd.DataFrame([{'country': r['country'].upper(), 'n': r['n'], 'SME_WIN rate': round(r['sme_win_rate'], 3),
                     'AUC within (a)': r['a_within']['auc'], 'AUC pooled incl. (b)': r['b_pooled_included']['auc'],
                     'AUC pooled excl. (c)': r['c_pooled_excluded']['auc'],
                     'F1m within (a)': r['a_within']['f1_macro'], 'F1m pooled incl. (b)': r['b_pooled_included']['f1_macro'],
                     'F1m pooled excl. (c)': r['c_pooled_excluded']['f1_macro'],
                     'F1m majority': r['majority_f1_macro']} for r in rows]).round(4)
print()
print(res.to_string(index=False))
print('\nMean AUC:  a =', round(res['AUC within (a)'].mean(), 4),
      ' b =', round(res['AUC pooled incl. (b)'].mean(), 4),
      ' c =', round(res['AUC pooled excl. (c)'].mean(), 4))

## 5. Optional: does telling the model the country help?
Pooled stratified k=10 with all `ISO_COUNTRY_CODE` dummies added to the 16 predictors.

In [ ]:
country_cols = [c for c in df_proc.columns if c.startswith('ISO_COUNTRY_CODE_')]
extra = [c for c in country_cols if c not in used_features_B2]
X_cty = np.hstack([X, df_proc[extra].astype(float).to_numpy()])
print('Features with country dummies:', X_cty.shape[1])

t0 = time.time()
op, oq = np.zeros(len(y), dtype=int), np.zeros(len(y))
for tr, te in skf.split(X_cty, y):
    op[te], oq[te] = fit_predict(X_cty[tr], y[tr], X_cty[te])
with_cty = scores(y, op, oq)
print(f'Pooled OOF WITH country dummies:    macro-F1={with_cty["f1_macro"]:.4f}  AUC={with_cty["auc"]:.4f}  ({(time.time()-t0)/60:.1f} min)')
print(f'Pooled OOF WITHOUT country dummies: macro-F1={overall["f1_macro"]:.4f}  AUC={overall["auc"]:.4f}')

## 6. Save

In [ ]:
out = {'overall_pooled': overall, 'overall_pooled_with_country_dummies': with_cty,
       'descriptive': desc.reset_index().to_dict(orient='records'),
       'per_country': rows}
with open('country_shift_results.json', 'w') as fh:
    json.dump(out, fh, indent=2, default=float)
print('Saved country_shift_results.json')
try:
    from google.colab import files
    files.download('country_shift_results.json')
except Exception as e:
    print('Download it from the Files panel:', e)